# tarea 4 - centros de objetos circulares

voy a buscar los centros de unas monedas usando OpenCV

usé la [imagen coins](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.coins) y el ejemplo de [HoughCircles de OpenCV](https://docs.opencv.org/4.x/da/d53/tutorial_py_houghcircles.html)

## primero veo la imagen

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

# uso esta imagen porque ya viene lista y tiene varias monedas
imagen_original = data.coins()

plt.figure(figsize=(9, 6))
plt.imshow(imagen_original, cmap='gray')
plt.title('Imagen original de monedas')
plt.axis('off')
plt.show()

imagen_original.shape

## le quito un poco de ruido

la imagen ya está en gris, nada más la suavizo y reviso los bordes

In [ ]:
# primero suavizo un poco el relieve de las monedas
imagen_suave = cv2.GaussianBlur(imagen_original, (9, 9), 2)
bordes = cv2.Canny(imagen_suave, 50, 120)

figura, ejes = plt.subplots(1, 3, figsize=(15, 5))
ejes[0].imshow(imagen_original, cmap='gray')
ejes[0].set_title('Original')
ejes[1].imshow(imagen_suave, cmap='gray')
ejes[1].set_title('Filtro gaussiano')
ejes[2].imshow(bordes, cmap='gray')
ejes[2].set_title('Bordes de Canny')

for eje in ejes:
    eje.axis('off')
plt.tight_layout()
plt.show()

## probando valores

pruebo tres valores de `param2` porque cambia bastante la cantidad de círculos

In [ ]:
umbrales = [24, 28, 32]
cantidades = []

# aquí fui cambiando solo este valor
for umbral in umbrales:
    circulos_prueba = cv2.HoughCircles(
        imagen_suave, cv2.HOUGH_GRADIENT, dp=1.2, minDist=22,
        param1=100, param2=umbral, minRadius=10, maxRadius=35
    )
    cantidad = 0 if circulos_prueba is None else circulos_prueba.shape[1]
    cantidades.append(cantidad)

plt.figure(figsize=(6, 4))
plt.plot(umbrales, cantidades, marker='o')
plt.xlabel('valor de param2')
plt.ylabel('círculos detectados')
plt.title('Sensibilidad al umbral de Hough')
plt.grid(alpha=.3)
plt.show()

umbrales, cantidades

## resultado

me quedé con 28, el círculo va en verde y el centro en rojo

In [ ]:
circulos = cv2.HoughCircles(
    imagen_suave, cv2.HOUGH_GRADIENT, dp=1.2, minDist=22,
    param1=100, param2=28, minRadius=10, maxRadius=35
)

imagen_resultado = cv2.cvtColor(imagen_original, cv2.COLOR_GRAY2RGB)
centros = []

if circulos is not None:
    circulos = np.round(circulos[0]).astype(int)
    for numero, (centro_x, centro_y, radio) in enumerate(circulos, start=1):
        # marco el borde y también el punto central
        cv2.circle(imagen_resultado, (centro_x, centro_y), radio, (0, 220, 0), 2)
        cv2.circle(imagen_resultado, (centro_x, centro_y), 3, (255, 0, 0), -1)
        cv2.putText(imagen_resultado, str(numero), (centro_x + 4, centro_y - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, .35, (255, 255, 0), 1)
        centros.append([centro_x, centro_y, radio])

plt.figure(figsize=(10, 7))
plt.imshow(imagen_resultado)
plt.title('Círculos y centros estimados')
plt.axis('off')
plt.show()

centros[:10]

## lo que salió

yo cuento 24 monedas en la foto y comparo eso con lo que encontró el código

no es una medida perfecta porque podría marcar algo que no es moneda y al mismo tiempo saltarse otra

también noté que `param2`, el tamaño de los radios y el contraste cambian mucho el resultado, con otra foto seguramente habría que moverlos otra vez

In [ ]:
monedas_visibles = 24  # yo las conté una vez directamente en la imagen
diferencia_conteo = abs(len(centros) - monedas_visibles)
coincidencia_conteo = max(0, 1 - diferencia_conteo / monedas_visibles)

len(centros), diferencia_conteo, round(coincidencia_conteo, 3)